In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
PDF_PATH = "data/Indian_Rental_Agreement_v2.pdf"
CHROMA_DIR = "chroma_store"
COLLECTION = "rag_collection"
TOP_K_RETRIEVE = 10
TOP_K_RERANK = 4

In [3]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
pages = reader.pages
print(f"Total pages: {len(pages)}")
first_page_text = pages[0].extract_text()
print(f"First page text:\n{first_page_text[:500]}")


Total pages: 4
First page text:
RESIDENTIAL RENTAL AGREEMENT
 
Tenant-Friendly Agreement as per Indian Contract Act, 1872 and Transfer of Property Act, 1882
PARTIES TO THE AGREEMENT
This Rental Agreement is entered into on 1st January 2025 between:
LANDLORD: Mr. Rajesh Kumar Sharma, aged 55 years, residing at Flat No. 201, Sunshine
Apartments, Banjara Hills, Hyderabad - 500034.
TENANT: Mr. Arun Venkat Reddy, aged 28 years, residing at Plot No. 45, Madhapur, Hyderabad -
500081.
PROPERTY DETAILS
Property Address: Flat No. 304, G


In [4]:
full_text = ""
for i, page in enumerate(pages):
    text = page.extract_text() or ""
    full_text += f"\n\n[Page {i + 1}]\n{text}"
print(f"Total characters extracted: {len(full_text):,}")
print(f"Approx tokens (/4): {len(full_text)//4:,}")

Total characters extracted: 6,298
Approx tokens (/4): 1,574


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(full_text)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"   Avg chunk length    : {sum(len(c) for c in chunks)//len(chunks)} chars")
print(f"\n--- Chunk #0 ---\n{chunks[0]}")
print(f"\n--- Chunk #1 (notice overlap with chunk #0) ---\n{chunks[1]}")

✅ Total chunks created: 17
   Avg chunk length    : 426 chars

--- Chunk #0 ---
[Page 1]
RESIDENTIAL RENTAL AGREEMENT
 
Tenant-Friendly Agreement as per Indian Contract Act, 1872 and Transfer of Property Act, 1882
PARTIES TO THE AGREEMENT
This Rental Agreement is entered into on 1st January 2025 between:
LANDLORD: Mr. Rajesh Kumar Sharma, aged 55 years, residing at Flat No. 201, Sunshine
Apartments, Banjara Hills, Hyderabad - 500034.
TENANT: Mr. Arun Venkat Reddy, aged 28 years, residing at Plot No. 45, Madhapur, Hyderabad -
500081.
PROPERTY DETAILS

--- Chunk #1 (notice overlap with chunk #0) ---
500081.
PROPERTY DETAILS
Property Address: Flat No. 304, Green Valley Residency, Kondapur, Hyderabad - 500084.
Property Type: 2BHK Residential Apartment
Carpet Area: 950 square feet approximately
RENT AND SECURITY DEPOSIT
Monthly Rent: Rs. 18,000 per month.
Security Deposit: Rs. 54,000 (3 months rent), refunded within 30 days of vacating after deducting
damages.
Rent Due Date: On or before 5t

In [6]:
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_index": i, "source": PDF_PATH} for i in range(len(chunks))]

print(f"Sample ID       : {chunk_ids[0]}")
print(f"Sample metadata : {metadatas[0]}")
print(f"Total IDs       : {len(chunk_ids)}")

Sample ID       : chunk_0
Sample metadata : {'chunk_index': 0, 'source': 'data/Indian_Rental_Agreement_v2.pdf'}
Total IDs       : 17


In [7]:
from sentence_transformers import SentenceTransformer

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
print(f" Bi-encoder loaded")
print(f"   Max seq length : {bi_encoder.max_seq_length}")
print(f"   Embedding dims : {bi_encoder.get_sentence_embedding_dimension()}")

/workspaces/RAG-Chatbot-Pipeline/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Bi-encoder loaded
   Max seq length : 256
   Embedding dims : 384


In [8]:
import numpy as np

sample_sentence = "This is a test sentence for RAG."
sample_embedding = bi_encoder.encode(sample_sentence)

print(f"Input sentence : '{sample_sentence}'")
print(f"Output shape   : {sample_embedding.shape}")
print(f"dtype          : {sample_embedding.dtype}")
print(f"L2 norm        : {np.linalg.norm(sample_embedding):.4f}")
print(f"\nFirst 10 dims  : {sample_embedding[:10].round(4)}")
print(f"Min / Max      : {sample_embedding.min():.4f} / {sample_embedding.max():.4f}")

Input sentence : 'This is a test sentence for RAG.'
Output shape   : (384,)
dtype          : float32
L2 norm        : 1.0000

First 10 dims  : [ 0.0019  0.1258  0.0743  0.0442 -0.0735  0.0207  0.0453 -0.005  -0.0186
  0.0073]
Min / Max      : -0.1618 / 0.1636


In [9]:
from sentence_transformers.util import cos_sim

sent_a = "What is the monthly rent?"
sent_b = "Monthly Rent: Rs. 18,000 per month."
sent_c = "The weather today is sunny and warm."

emb_a = bi_encoder.encode(sent_a)
emb_b = bi_encoder.encode(sent_b)
emb_c = bi_encoder.encode(sent_c)

print(f"Similarity (A vs B — related)   : {cos_sim(emb_a, emb_b).item():.4f}")
print(f"Similarity (A vs C — unrelated) : {cos_sim(emb_a, emb_c).item():.4f}")

Similarity (A vs B — related)   : 0.7968
Similarity (A vs C — unrelated) : -0.0039


In [10]:
import time

print(f"Embedding {len(chunks)} chunks...")
t0 = time.time()

embeddings = bi_encoder.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

elapsed = time.time() - t0
print(f"\n Done in {elapsed:.1f}s")
print(f"   Embeddings matrix shape : {embeddings.shape}")
print(f"   = {embeddings.shape[0]} chunks × {embeddings.shape[1]} dims")

Embedding 17 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


 Done in 1.0s
   Embeddings matrix shape : (17, 384)
   = 17 chunks × 384 dims


In [11]:
import chromadb

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    chroma_client.delete_collection(name=COLLECTION)
    print("🗑️  Old collection deleted.")
except:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

print(f"Collection '{COLLECTION}' created.")

2026-04-08 13:10:11.179266867 [W:onnxruntime:Default, device_discovery.cc:132 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection 'rag_collection' created.


In [12]:
BATCH = 100
for start in range(0, len(chunks), BATCH):
    end = min(start + BATCH, len(chunks))
    collection.add(
        ids=chunk_ids[start:end],
        documents=chunks[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadatas[start:end],
    )

print(f"✅ Stored {collection.count()} chunks in ChromaDB.")

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ Stored 17 chunks in ChromaDB.


In [13]:
peek = collection.peek(limit=2)

print("=== ChromaDB Internal Structure ===")
print(f"Keys in response : {list(peek.keys())}")
print(f"\n-- IDs --")
print(peek["ids"])
print(f"\n-- Metadata --")
print(peek["metadatas"])
print(f"\n-- Document (chunk text) --")
print(peek["documents"][0][:200], "...")
print(f"\n-- Embedding (first 8 dims of chunk_0) --")
print(peek["embeddings"][0][:8])

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


=== ChromaDB Internal Structure ===
Keys in response : ['ids', 'embeddings', 'documents', 'uris', 'data', 'metadatas', 'included']

-- IDs --
['chunk_0', 'chunk_1']

-- Metadata --
[{'chunk_index': 0, 'source': 'data/Indian_Rental_Agreement_v2.pdf'}, {'chunk_index': 1, 'source': 'data/Indian_Rental_Agreement_v2.pdf'}]

-- Document (chunk text) --
[Page 1]
RESIDENTIAL RENTAL AGREEMENT
 
Tenant-Friendly Agreement as per Indian Contract Act, 1872 and Transfer of Property Act, 1882
PARTIES TO THE AGREEMENT
This Rental Agreement is entered into on  ...

-- Embedding (first 8 dims of chunk_0) --
[ 0.00566314  0.02859611 -0.02659872  0.01120191 -0.07603963  0.0557344
 -0.01666832 -0.05278401]


In [14]:
USER_QUERY = "What is the monthly rent and security deposit?"

query_embedding = bi_encoder.encode(USER_QUERY).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=TOP_K_RETRIEVE,
    include=["documents", "distances", "metadatas"],
)

retrieved_docs      = results["documents"][0]
retrieved_distances = results["distances"][0]
retrieved_metadata  = results["metadatas"][0]

print(f"Query           : '{USER_QUERY}'")
print(f"Retrieved top-{TOP_K_RETRIEVE}")
print("="*60)
for i, (doc, dist, meta) in enumerate(zip(retrieved_docs, retrieved_distances, retrieved_metadata)):
    print(f"\n[Rank {i+1}] chunk_index={meta['chunk_index']}  distance={dist:.4f}")
    print(doc[:200] + "...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query           : 'What is the monthly rent and security deposit?'
Retrieved top-10

[Rank 1] chunk_index=3  distance=0.4654
[Page 2]
The Tenant has the full right to peaceful enjoyment of the property. The Landlord shall NOT disturb
the Tenant unnecessarily. The Landlord must give at least 24 hours advance notice before vi...

[Rank 2] chunk_index=10  distance=0.4685
Always pay rent via bank transfer, UPI or cheque rather than cash. This creates a digital payment
trail. Save all rent receipts or bank statements as proof of payment. This is your protection if the
L...

[Rank 3] chunk_index=14  distance=0.4732
To ensure full recovery of your security deposit: Give proper 2 months notice in writing, clean the
property thoroughly before vacating, repair any damages you caused, return all keys and access
cards...

[Rank 4] chunk_index=13  distance=0.5030
Suggestion 6: Understand Utility Bills
Electricity and water bills are in the Tenant's name and responsibility. Always check the meter
re

In [16]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print(" Cross-encoder loaded")

pairs = [(USER_QUERY, doc) for doc in retrieved_docs]
ce_scores = cross_encoder.predict(pairs)

print(f"\nCross-encoder raw scores (before sorting):")
for i, score in enumerate(ce_scores):
    print(f"  Bi-encoder rank {i+1}  →  CE score: {score:.4f}")

 Cross-encoder loaded

Cross-encoder raw scores (before sorting):
  Bi-encoder rank 1  →  CE score: -0.4807
  Bi-encoder rank 2  →  CE score: -6.6615
  Bi-encoder rank 3  →  CE score: -4.0387
  Bi-encoder rank 4  →  CE score: -6.1787
  Bi-encoder rank 5  →  CE score: -4.8263
  Bi-encoder rank 6  →  CE score: -11.2168
  Bi-encoder rank 7  →  CE score: 5.1030
  Bi-encoder rank 8  →  CE score: -5.3100
  Bi-encoder rank 9  →  CE score: -7.9281
  Bi-encoder rank 10  →  CE score: -9.5426


In [18]:
import pandas as pd

ranked_indices  = sorted(range(len(ce_scores)), key=lambda i: ce_scores[i], reverse=True)
reranked_docs   = [retrieved_docs[i]  for i in ranked_indices[:TOP_K_RERANK]]
reranked_scores = [ce_scores[i]       for i in ranked_indices[:TOP_K_RERANK]]

comparison = []
for new_rank, old_rank in enumerate(ranked_indices[:TOP_K_RERANK]):
    comparison.append({
        "New Rank"              : new_rank + 1,
        "Old BI Rank"           : old_rank + 1,
        "CE Score"              : round(float(ce_scores[old_rank]), 4),
        "BI Distance"           : round(retrieved_distances[old_rank], 4),
        "Chunk (first 80 chars)": retrieved_docs[old_rank][:80] + "..."
    })

df = pd.DataFrame(comparison)
print("=== Reranking Result ===")
print(df.to_string(index=False))
print(f"\n Kept {TOP_K_RERANK} of {TOP_K_RETRIEVE} chunks for context.")

=== Reranking Result ===
 New Rank  Old BI Rank  CE Score  BI Distance                                                                Chunk (first 80 chars)
        1            7    5.1030       0.5912 500081.\nPROPERTY DETAILS\nProperty Address: Flat No. 304, Green Valley Residency,...
        2            1   -0.4807       0.4654  [Page 2]\nThe Tenant has the full right to peaceful enjoyment of the property. Th...
        3            3   -4.0387       0.4732   To ensure full recovery of your security deposit: Give proper 2 months notice in...
        4            5   -4.8263       0.5713 [Page 3]\nSuggestion 1: Document Everything\nTake photographs and videos of the en...

 Kept 4 of 10 chunks for context.


In [19]:
def build_prompt(query: str, context_chunks: list) -> str:
    context = "\n\n---\n\n".join(
        [f"[Context {i+1}]:\n{chunk}" for i, chunk in enumerate(context_chunks)]
    )
    return f"""You are a precise document assistant. Answer the user's question using ONLY the context provided below.
If the answer is not found in the context, say: "I could not find relevant information in the document."
Do not use any outside knowledge.

=== CONTEXT ===
{context}

=== QUESTION ===
{query}

=== ANSWER ==="""

prompt = build_prompt(USER_QUERY, reranked_docs)

print(f"Prompt length : {len(prompt)} chars (~{len(prompt)//4} tokens)")
print("\n--- Prompt Preview (first 800 chars) ---")
print(prompt[:800])

Prompt length : 2262 chars (~565 tokens)

--- Prompt Preview (first 800 chars) ---
You are a precise document assistant. Answer the user's question using ONLY the context provided below.
If the answer is not found in the context, say: "I could not find relevant information in the document."
Do not use any outside knowledge.

=== CONTEXT ===
[Context 1]:
500081.
PROPERTY DETAILS
Property Address: Flat No. 304, Green Valley Residency, Kondapur, Hyderabad - 500084.
Property Type: 2BHK Residential Apartment
Carpet Area: 950 square feet approximately
RENT AND SECURITY DEPOSIT
Monthly Rent: Rs. 18,000 per month.
Security Deposit: Rs. 54,000 (3 months rent), refunded within 30 days of vacating after deducting
damages.
Rent Due Date: On or before 5th of every month.
Late Payment Penalty: Rs. 500 per day after 10th of the month.
DURATION

---

[Context 2]:
[Page 2]
The Tenant has


In [20]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def generate_answer(prompt: str, model: str = "llama-3.1-8b-instant") -> str:
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful document assistant."},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.1,
        max_tokens=512,
    )
    return response.choices[0].message.content

answer = generate_answer(prompt)

print(f"Query  : {USER_QUERY}")
print(f"\nAnswer :\n{answer}")

Query  : What is the monthly rent and security deposit?

Answer :
The monthly rent is Rs. 18,000 per month. 
The security deposit is Rs. 54,000.


In [21]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from typing import TypedDict

class RAGState(TypedDict):
    query         : str
    query_embedding: list
    retrieved_docs : list
    retrieved_distances: list
    reranked_docs  : list
    ce_scores      : list
    prompt         : str
    answer         : str

def retrieve_node(state: RAGState) -> RAGState:
    q_embedding = bi_encoder.encode(state["query"]).tolist()
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=TOP_K_RETRIEVE,
        include=["documents", "distances"],
    )
    return {
        **state,
        "query_embedding"    : q_embedding,
        "retrieved_docs"     : results["documents"][0],
        "retrieved_distances": results["distances"][0],
    }

def rerank_node(state: RAGState) -> RAGState:
    pairs   = [(state["query"], doc) for doc in state["retrieved_docs"]]
    scores  = cross_encoder.predict(pairs)
    ranked  = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return {
        **state,
        "reranked_docs": [state["retrieved_docs"][i] for i in ranked[:TOP_K_RERANK]],
        "ce_scores"    : [float(scores[i])           for i in ranked[:TOP_K_RERANK]],
    }

def generate_node(state: RAGState) -> RAGState:
    prompt = build_prompt(state["query"], state["reranked_docs"])
    answer = generate_answer(prompt)
    return {
        **state,
        "prompt": prompt,
        "answer": answer,
    }

graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("rerank",   rerank_node)
graph.add_node("generate", generate_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "rerank")
graph.add_edge("rerank",   "generate")
graph.add_edge("generate", END)

rag_app = graph.compile()
print("LangGraph RAG pipeline compiled.")
print("   Flow: retrieve → rerank → generate → END")


LangGraph RAG pipeline compiled.
   Flow: retrieve → rerank → generate → END


In [22]:
initial_state = {
    "query"             : "What is the monthly rent and security deposit?",
    "query_embedding"   : [],
    "retrieved_docs"    : [],
    "retrieved_distances": [],
    "reranked_docs"     : [],
    "ce_scores"         : [],
    "prompt"            : "",
    "answer"            : "",
}

final_state = rag_app.invoke(initial_state)

print(f"Query  : {final_state['query']}")
print(f"\nAnswer :\n{final_state['answer']}")
print(f"\nCE Scores (top-{TOP_K_RERANK}): {[round(s, 4) for s in final_state['ce_scores']]}")

Query  : What is the monthly rent and security deposit?

Answer :
The monthly rent is Rs. 18,000 per month. 
The security deposit is Rs. 54,000.

CE Scores (top-4): [5.103, -0.4807, -4.0387, -4.8263]


In [23]:
print("=== RAG Chatbot (type 'exit' to quit) ===")
print("PDF: Indian Rental Agreement\n")

while True:
    query = input("You: ").strip()
    
    if query.lower() in ("exit", "quit", ""):
        print("Goodbye!")
        break
    
    initial_state = {
        "query"              : query,
        "query_embedding"    : [],
        "retrieved_docs"     : [],
        "retrieved_distances": [],
        "reranked_docs"      : [],
        "ce_scores"          : [],
        "prompt"             : "",
        "answer"             : "",
    }
    
    final_state = rag_app.invoke(initial_state)
    print(f"\nBot: {final_state['answer']}\n")

=== RAG Chatbot (type 'exit' to quit) ===
PDF: Indian Rental Agreement


Bot: The security deposit is Rs. 54,000.

Goodbye!
